[Lab README](README.md)

# Lab 1: From text to a knowledge graph

Thirty hotel FAQ documents go in as plain text. What comes out is a typed graph:
`Hotel`, `Room`, `Amenity`, `Policy`, and `Service` nodes joined by the four
relationships every later lab queries, a 1024-dimensional embedding on every
chunk, and the two Neo4j indexes that retrieval runs against.

Nothing in this repository ships pre-embedded. You run the extraction yourself,
so what you take into Lab 2 is what your models actually produced.

| | |
|---|---|
| **Neo4j owns** | Aura, `SimpleKGPipeline`, the pinned schema, the vector index `hotel_chunk_embeddings`, the full-text index `hotel_chunk_fulltext`, and three uniqueness constraints |
| **AWS owns** | Bedrock `us.anthropic.claude-sonnet-5` for entity and relationship extraction, and Amazon Nova 2 Multimodal Embeddings at 1024 dimensions for chunk vectors |

## Before you start

Finish the Lab 0 checklist first. This notebook needs `NEO4J_URI`,
`NEO4J_USERNAME`, and `NEO4J_PASSWORD` in the repo-root `.env`, plus AWS
credentials that may invoke both Bedrock models in `AWS_REGION`.

The 30-document build takes about 15 minutes and makes real Bedrock calls. It is
idempotent: run it against a graph that is already complete and it verifies
rather than rebuilds, so re-running this notebook is cheap.

In [ ]:
# At an AWS event: dependencies are pre-installed. Run this cell as-is.
# Self-paced: uncomment the line below first.
# !pip install -r requirements.txt

import zipfile
from pathlib import Path

# Only the zip is committed. `data/` is gitignored, so a fresh clone has no
# corpus until this runs. Extracting twice would be wasted work, so it skips
# when the documents are already there.
DATA_DIR = Path("data")
CORPUS_ZIP = Path("hotel-faqs.zip")

extracted = sorted(DATA_DIR.glob("*.txt"))
if extracted:
    print(f"Corpus already extracted: {len(extracted)} documents in {DATA_DIR}/")
elif CORPUS_ZIP.exists():
    with zipfile.ZipFile(CORPUS_ZIP) as archive:
        archive.extractall(DATA_DIR)
    print(f"Extracted {len(sorted(DATA_DIR.glob('*.txt')))} documents into {DATA_DIR}/")
else:
    print(
        f"{CORPUS_ZIP} is not here. Run this notebook from 01-graph-build/, "
        "which is where both the corpus and the build scripts live."
    )

print("Environment ready")

## 1. Check the environment

Every module that opens a Neo4j driver imports `workshop.graph_connection`,
which raises at import when `NEO4J_PASSWORD` is unset. So this cell reads the
environment directly and sets a flag. The cells that connect import their
modules later, inside that guard.

In [ ]:
import os

import boto3
from dotenv import load_dotenv

load_dotenv()

REQUIRED_NEO4J_VARS = ("NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD")
missing = [name for name in REQUIRED_NEO4J_VARS if not os.environ.get(name)]
has_aws = boto3.Session().get_credentials() is not None

BUILD_READY = not missing and has_aws

# "lite" is the 30-document sample the workshop runs on, roughly 15 minutes.
# "full" is all 300 documents and takes about 2 hours.
MODE = "lite"

# The command-line build takes --rebuild to discard a graph that already reports
# ready. Set this to True for the same effect.
REBUILD = False

if missing:
    print(f"Neo4j is not configured: set {', '.join(missing)} in the repo-root .env")
if not has_aws:
    print("No AWS credentials found, so Bedrock extraction and embedding cannot run")

if BUILD_READY:
    print(f"Ready to build. mode={MODE} rebuild={REBUILD}")
    print(f"AWS_REGION={os.environ.get('AWS_REGION', 'us-east-1')}")
else:
    print("\nThe cells below will skip. Finish Lab 0, then run this notebook again.")

## 2. Pin the extraction schema

`SimpleKGPipeline` will happily run without a schema. It then asks the model to
invent labels for each chunk, and one document yields an `Address` node while
the next yields `Location`. Lab 3 hands its agent a retrieval tool that promises
a fixed contract, so the graph has to honour one.

The schema below is that contract. `additional_node_types` and its two siblings
are `False`, which is what turns the schema from a suggestion into a rule.

In [ ]:
from IPython.display import HTML, display

from workshop.graph_schema import GRAPH_SCHEMA

# The descriptions are not decoration: they travel to Claude with the schema and
# are what keep an address a property instead of a node.
node_rows = "".join(
    f"<tr><td><strong>{node['label']}</strong></td>"
    f"<td>{node.get('description', '')}</td>"
    f"<td>{', '.join(prop['name'] for prop in node['properties'])}</td></tr>"
    for node in GRAPH_SCHEMA["node_types"]
)
property_rows = "".join(
    f"<tr><td><strong>{node['label']}</strong>.{prop['name']}</td>"
    f"<td>{prop['description']}</td></tr>"
    for node in GRAPH_SCHEMA["node_types"]
    for prop in node["properties"]
    if prop.get("description")
)
pattern_rows = "".join(
    f"<tr><td><strong>{source}</strong></td><td>-[:{relationship}]-&gt;</td>"
    f"<td><strong>{target}</strong></td></tr>"
    for source, relationship, target in GRAPH_SCHEMA["patterns"]
)
display(HTML(
    "<table><thead><tr><th>Node</th><th>What the schema says it is</th>"
    "<th>Properties</th></tr></thead>"
    f"<tbody>{node_rows}</tbody></table>"
    "<table><thead><tr><th>Property</th><th>Extraction instruction</th></tr>"
    f"</thead><tbody>{property_rows}</tbody></table>"
    "<table><thead><tr><th>From</th><th>Relationship</th><th>To</th></tr>"
    f"</thead><tbody>{pattern_rows}</tbody></table>"
))

for key in ("additional_node_types", "additional_relationship_types", "additional_patterns"):
    print(f"{key}: {GRAPH_SCHEMA[key]}")

## 3. Select the source documents

The lite sample is not the first 30 filenames. Lab 2 asks about Paris and Cairo
by name, and an alphabetical cut stops at Boston, so the selection takes every
Paris and Cairo document first and fills the rest round-robin across the other
cities.

Five documents are load-bearing for later labs. If the selection ever loses one,
the build refuses to start rather than producing a graph whose demo questions
quietly return nothing.

In [ ]:
if not BUILD_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    # Imported here rather than at the top of the notebook: this module reaches
    # `workshop.graph_connection`, which raises when Neo4j is unconfigured.
    from prepare_graph import selected_paths
    from workshop.retrieval_setup import missing_source_fixtures

    paths = selected_paths(MODE)
    assert paths, (
        f"no source documents in {DATA_DIR.resolve()}. The corpus is not "
        "extracted: run the first cell of this notebook, or "
        "`unzip -q -o hotel-faqs.zip -d data/` from 01-graph-build/"
    )

    paris = sum(1 for path in paths if "-paris-" in path.name)
    cairo = sum(1 for path in paths if "-cairo-" in path.name)

    print(f"{len(paths)} documents selected in {MODE} mode")
    print(f"Paris: {paris}, Cairo: {cairo}, other cities: {len(paths) - paris - cairo}")

    absent = missing_source_fixtures(paths)
    assert not absent, f"demo-critical source documents are missing: {absent}"
    print("All demo-critical source documents are in the selection")

## 4. Ask whether the graph is already built

This is the check that makes the notebook re-runnable. It creates the two
retrieval indexes if they are absent, then reports what the graph holds and
which demo-critical facts are missing. An empty list of problems means Lab 2 can
already run and there is nothing to rebuild.

In [ ]:
if not BUILD_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    from graph_builder import connect
    from workshop.retrieval_setup import (
        ReadinessError,
        ensure_retrieval_indexes,
        report_readiness,
    )

    driver = connect()
    driver.verify_connectivity()
    print("Connected to Neo4j.")

    # An index that exists but does not match the embedding contract is not a
    # rebuild away. Its vectors are the wrong shape, so a build would spend
    # fifteen minutes writing embeddings the index cannot serve. The script
    # path in prepare_graph.py stops here too.
    INDEX_ERROR = None
    problems = []
    try:
        ensure_retrieval_indexes(driver)
    except ReadinessError as exc:
        INDEX_ERROR = str(exc)
    else:
        problems = report_readiness(driver, expected_documents=len(paths))

    NEEDS_BUILD = not INDEX_ERROR and (REBUILD or bool(problems))

    if INDEX_ERROR:
        print(f"\n❌ {INDEX_ERROR}")
        print(
            "\nThe existing index does not match the embedding contract in "
            "workshop.retrieval_contract. Drop it in Neo4j, then re-run this "
            "cell. Do not edit the contract to match the index: the vectors "
            "themselves would still be wrong for Lab 2."
        )
    elif problems:
        print("\nThe graph is not ready:")
        for problem in problems:
            print(f"  - {problem}")

    if NEEDS_BUILD:
        print("\nThe next cell will build the graph.")
    elif not INDEX_ERROR:
        print("\nThe graph is already complete. The next cell will skip the build.")

## 5. Run the extraction

`SimpleKGPipeline` splits each document into chunks, sends each chunk to Claude
with the pinned schema, writes the extracted nodes and relationships, and asks
Nova for the chunk embedding. One chunk holds a whole hotel, so a hotel's name,
address, and rating are extracted in the same prompt as its rooms and amenities.

The build canaries three documents first and checks what came back before
spending 15 minutes on the rest. If the canary finds an off-schema label or no
complete hotel, it clears the graph and stops, because a graph that is wrong in
that way looks plausible right up until Lab 2 asks it a question.

In [ ]:
if not BUILD_READY:
    print("Skipping: no Neo4j or AWS configuration.")
elif INDEX_ERROR:
    print(
        "Skipping: the retrieval index does not match the embedding contract. "
        "Fix that first, because this build would write vectors the index "
        "cannot serve."
    )
elif not NEEDS_BUILD:
    from graph_builder import report

    print("Skipping the build: the graph already reports ready. Set REBUILD = True to force one.")
    # The report runs whether or not a build ran, so a participant arriving at
    # an already-built graph still sees the acceptance queries Lab 2 depends on.
    report(driver)
else:
    from graph_builder import run_build

    title = "LITE BUILD" if MODE == "lite" else "FULL BUILD"
    exit_code = await run_build(paths, title)
    assert exit_code == 0, "the build did not complete; read the output above"

## 6. Verify the two retrieval indexes

Lab 2 opens onto whatever this leaves behind, and the failure it cares about is
silent. A vector index built at the wrong dimension or the wrong similarity
function does not raise, it just returns the wrong neighbours. So the check is
against the contract, not against existence.

In [ ]:
if not BUILD_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    from workshop.retrieval_contract import (
        CHUNK_FULLTEXT_INDEX,
        CHUNK_VECTOR_INDEX,
        EMBEDDING_DIMENSIONS,
    )
    from workshop.retrieval_setup import fixture_problems, verify_retrieval_indexes

    verify_retrieval_indexes(driver)
    print(f"{CHUNK_VECTOR_INDEX} is ONLINE at {EMBEDDING_DIMENSIONS} dimensions, cosine")
    print(f"{CHUNK_FULLTEXT_INDEX} is ONLINE over :Chunk(text)")

    remaining = fixture_problems(driver)
    assert not remaining, f"demo-critical fixtures are still missing: {remaining}"
    print("Every demo-critical graph fixture is present")

## 7. Seed the fixtures Labs 4 and 5 depend on

Extraction gives each hotel a name and an address, which is enough to retrieve
against. It does not give them a stable identifier, and the reservation write in
Lab 4 needs one: an agent that books against a hotel found by name books against
whatever the model spelled that day.

This step is graph-owned data rather than extracted data. It adds three
uniqueness constraints, stamps the committed opaque ID onto each of the two
Cairo fixture hotels, and writes the `max_guests` rule that Lab 4 enforces on
the write path. It is a `MERGE` throughout, so running it twice changes nothing.

In [ ]:
if not BUILD_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    from workshop import contracts
    from workshop.graph_setup import apply_lab4_fixtures, load_manifest, readiness_problems

    database = os.environ.get("NEO4J_DATABASE", "neo4j")
    manifest = load_manifest()

    blockers = apply_lab4_fixtures(driver, database, manifest)
    assert not blockers, f"the fixtures could not be applied: {blockers}"

    outstanding = readiness_problems(driver, database, manifest)
    assert not outstanding, f"the graph is not ready for Lab 4: {outstanding}"

    # Read the three facts back out of the graph rather than restating what the
    # code above intended to write.
    with driver.session(database=database) as session:
        stamped = session.run(
            "MATCH (h:Hotel) WHERE h.hotel_id IS NOT NULL RETURN count(h) AS count"
        ).single()["count"]
        constraints = sorted(
            record["name"]
            for record in session.run("SHOW CONSTRAINTS YIELD name RETURN name")
            if record["name"].startswith("demo06_")
        )
        rule = session.run(
            """
            MATCH (rule:Rule {rule_id: $rule_id})
            RETURN rule.rule_type AS rule_type,
                   rule.max_guests AS max_guests,
                   rule.enabled AS enabled
            """,
            rule_id=contracts.MAX_GUESTS_RULE_ID,
        ).single()

    print(f"Hotels carrying a fixture ID: {stamped} of {len(manifest.hotels)} in the manifest")
    print(f"Constraints in the graph: {', '.join(constraints)}")
    print(
        f"Rule {contracts.MAX_GUESTS_RULE_ID}: {rule['rule_type']}, "
        f"max_guests={rule['max_guests']}, enabled={rule['enabled']}"
    )

## 8. What you built

Look at the counts the readiness report printed above rather than taking this
table on faith.

| | |
|---|---|
| `:Document` and `:Chunk` | one of each per source document, each chunk carrying a 1024-dimensional Nova embedding |
| `:Hotel`, `:Room`, `:Amenity`, `:Policy`, `:Service` | extracted by Claude against the pinned schema, and nothing outside it |
| `hotel_chunk_embeddings` | the vector index Lab 2's semantic retrievers search |
| `hotel_chunk_fulltext` | the full-text index that keeps exact identifiers findable |
| Fixture IDs, constraints, and the `max_guests` rule | the graph-owned data Lab 4 writes against |

Nothing here was pre-computed. The same 30 documents run twice produce slightly
different extractions, which is the honest starting point for the rest of the
workshop: the graph is only as good as what you can verify about it.

**Next:** Lab 2 runs four retrieval patterns over this graph and closes on a
question it cannot answer.

In [ ]:
if BUILD_READY:
    driver.close()
    print("Connection closed.")
else:
    print("Skipping: no Neo4j or AWS configuration.")